# LAK-12 — Copy-on-write vs merge-on-read, across Iceberg *and* Hudi

Both Iceberg and Hudi support two write strategies for updates/deletes:

- **Copy-on-write (CoW):** rewrite the whole base file that contains a changed row. Cheap reads, expensive writes.
- **Merge-on-read (MoR):** leave the base file, write a small side file (Iceberg: a **delete file** + a tiny data file; Hudi: a compact **log file**). Cheap writes, readers reconcile at query time.

We run **the same 1-row `MERGE` four ways** — Iceberg CoW, Iceberg MoR, Hudi CoW, Hudi MoR — and measure the cost.

**The measurement that matters is BYTES, not file counts.** A 1-row change can leave the same *number* of files while writing wildly different *bytes* — that byte delta is the write-amplification you're trading away. This notebook makes that visible.

> Prereqs: `make up`. Iceberg lands under `s3://warehouse/iceberg/`, Hudi under `s3://warehouse/hudi/`.

In [1]:
from common.spark_session import spark
from common.table_meta import hudi_table_health as hudi_health, wipe_prefix, s3_client, split_s3

ICE_COW = "iceberg_catalog.default.lak12_ice_cow"
ICE_MOR = "iceberg_catalog.default.lak12_ice_mor"
HUDI_COW = "spark_catalog.default.lak12_hudi_cow";  HUDI_COW_LOC = "s3a://warehouse/hudi/lak12_cow"
HUDI_MOR = "spark_catalog.default.lak12_hudi_mor";  HUDI_MOR_LOC = "s3a://warehouse/hudi/lak12_mor"

# Clean start. Iceberg (Hadoop catalog) DROP purges files; Hudi DROP does not,
# so we wipe the Hudi prefixes directly. Idempotent — safe to re-run.
for t in (ICE_COW, ICE_MOR, HUDI_COW, HUDI_MOR):
    spark.sql(f"DROP TABLE IF EXISTS {t}")
for loc in (HUDI_COW_LOC, HUDI_MOR_LOC):
    wipe_prefix(loc)

R = {}  # results per config, filled below, read by the comparison cell
print("reset done")

reset done


## The shared MERGE + how we inspect each format

One `MERGE` (ship order #1), replayed against all four tables. The inspection functions below are the **real technique** for each format — Iceberg via its metadata tables (`.all_data_files`, `.files`, `.snapshots`), Hudi by listing the files on its S3 prefix. Nothing here is hidden in a helper; read them.

In [2]:
# The one MERGE we replay four times (cast to BIGINT to match target columns).
MERGE_SRC = """
    SELECT CAST(1 AS BIGINT) AS order_id, CAST(100 AS BIGINT) AS customer_id,
           50.0 AS amount, 'SHIPPED' AS status, current_timestamp() AS updated_at
"""

def seed(table):
    # REPARTITION(1) packs the 3-row seed into ONE write task so before/after
    # counts stay legible. (This is a Spark write-planning hint — Iceberg emits
    # one file; Hudi lands one file group here simply because the data is tiny.)
    spark.sql(f"""
        INSERT INTO {table}
        SELECT /*+ REPARTITION(1) */ *
        FROM VALUES
            (CAST(1 AS BIGINT), CAST(100 AS BIGINT), 50.0,  'NEW',  current_timestamp()),
            (CAST(2 AS BIGINT), CAST(101 AS BIGINT), 75.0,  'PAID', current_timestamp()),
            (CAST(3 AS BIGINT), CAST(102 AS BIGINT), 120.0, 'NEW',  current_timestamp())
        AS t(order_id, customer_id, amount, status, updated_at)
    """)

def do_merge(table):
    spark.sql(f"""
        MERGE INTO {table} t USING ({MERGE_SRC}) s
        ON t.order_id = s.order_id
        WHEN MATCHED     THEN UPDATE SET status = s.status, updated_at = s.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)

# ── Iceberg inspection: real metadata-table SQL ──────────────────────────────
def ice_files(table):
    return int(spark.sql(f"SELECT COUNT(*) n FROM {table}.all_data_files").first()["n"])

def ice_bytes(table):
    # Sum bytes of every data file ever written (live or superseded) — mirrors
    # 'all parquet on the Hudi prefix', so a CoW rewrite shows up on both sides.
    return int(spark.sql(
        f"SELECT COALESCE(SUM(file_size_in_bytes),0) b FROM {table}.all_data_files").first()["b"])

def ice_delete_files(table):
    rows = {int(r["content"]): int(r["n"]) for r in spark.sql(
        f"SELECT content, COUNT(*) n FROM {table}.files GROUP BY content").collect()}
    return rows.get(1, 0) + rows.get(2, 0)  # 1=position deletes, 2=equality deletes

def ice_snapshot_summary(table):
    r = spark.sql(f"""
        SELECT summary['added-data-files'] a, summary['deleted-data-files'] d,
               summary['added-delete-files'] x
        FROM {table}.snapshots ORDER BY committed_at DESC LIMIT 1""").first()
    return {"added": int(r["a"] or 0), "deleted": int(r["d"] or 0), "added_deletes": int(r["x"] or 0)}

# ── Hudi inspection: list the files on the prefix (the parallel technique) ────
def hudi_show_files(loc, label):
    s3 = s3_client(); bucket, prefix = split_s3(loc)
    print(f"  {label} — data/log files on {loc}:")
    for o in sorted(s3.list_objects_v2(Bucket=bucket, Prefix=prefix).get("Contents", []),
                    key=lambda x: x["Key"]):
        k = o["Key"]
        if "/.hoodie/" in f"/{k}":
            continue
        if k.endswith(".parquet") or ".log." in k:
            print(f"     {k.rsplit('/', 1)[-1]:<48} {o['Size']:>7} B")

print("tools defined")

tools defined


## Config 1 — Iceberg copy-on-write (the default)

`write.*.mode = copy-on-write`: the 1-row `MERGE` rewrites the whole data file.

In [3]:
spark.sql(f"""
CREATE TABLE {ICE_COW} (
    order_id BIGINT, customer_id BIGINT, amount DOUBLE, status STRING, updated_at TIMESTAMP
) USING iceberg TBLPROPERTIES (
    'format-version'='2',
    'write.delete.mode'='copy-on-write',
    'write.update.mode'='copy-on-write',
    'write.merge.mode'='copy-on-write'
)
""")
seed(ICE_COW)
fb, bb = ice_files(ICE_COW), ice_bytes(ICE_COW)
do_merge(ICE_COW)
R["Iceberg CoW"] = dict(fb=fb, fa=ice_files(ICE_COW), bb=bb, ba=ice_bytes(ICE_COW),
                        xtra=ice_delete_files(ICE_COW), kind="delete", snap=ice_snapshot_summary(ICE_COW))
print("Iceberg CoW:", R["Iceberg CoW"])

Iceberg CoW: {'fb': 1, 'fa': 2, 'bb': 1613, 'ba': 3252, 'xtra': 0, 'kind': 'delete', 'snap': {'added': 1, 'deleted': 1, 'added_deletes': 0}}


## Config 2 — Iceberg merge-on-read

`write.*.mode = merge-on-read`: the same `MERGE` writes a **delete file** (+ a tiny data file for the new value) instead of rewriting the base file.

In [4]:
spark.sql(f"""
CREATE TABLE {ICE_MOR} (
    order_id BIGINT, customer_id BIGINT, amount DOUBLE, status STRING, updated_at TIMESTAMP
) USING iceberg TBLPROPERTIES (
    'format-version'='2',
    'write.delete.mode'='merge-on-read',
    'write.update.mode'='merge-on-read',
    'write.merge.mode'='merge-on-read'
)
""")
seed(ICE_MOR)
fb, bb = ice_files(ICE_MOR), ice_bytes(ICE_MOR)
do_merge(ICE_MOR)
R["Iceberg MoR"] = dict(fb=fb, fa=ice_files(ICE_MOR), bb=bb, ba=ice_bytes(ICE_MOR),
                        xtra=ice_delete_files(ICE_MOR), kind="delete", snap=ice_snapshot_summary(ICE_MOR))
print("Iceberg MoR:", R["Iceberg MoR"])

Iceberg MoR: {'fb': 1, 'fa': 2, 'bb': 1613, 'ba': 3154, 'xtra': 1, 'kind': 'delete', 'snap': {'added': 1, 'deleted': 0, 'added_deletes': 1}}


## Config 3 — Hudi copy-on-write

LAK-11's pattern: `type='cow'` rewrites the base Parquet. We list the files so you see the rewritten slice (and the stale one) directly.

In [5]:
spark.sql(f"""
CREATE TABLE {HUDI_COW} (
    order_id BIGINT, customer_id BIGINT, amount DOUBLE, status STRING, updated_at TIMESTAMP
) USING hudi LOCATION '{HUDI_COW_LOC}' TBLPROPERTIES (
    'primaryKey'='order_id', 'preCombineField'='updated_at', 'type'='cow'
)
""")
seed(HUDI_COW)
h0 = hudi_health(HUDI_COW_LOC)
do_merge(HUDI_COW)
h1 = hudi_health(HUDI_COW_LOC)
R["Hudi CoW"] = dict(fb=h0["parquet_files_on_disk"], fa=h1["parquet_files_on_disk"],
                     bb=h0["total_bytes"], ba=h1["total_bytes"], xtra=h1["log_files"], kind="log")
hudi_show_files(HUDI_COW_LOC, "Hudi CoW after merge")
print("Hudi CoW:", R["Hudi CoW"])

  Hudi CoW after merge — data/log files on s3a://warehouse/hudi/lak12_cow:
     c9176329-eae5-4e14-a004-4ae66270f7bd-0_0-227-701_20260721050957422.parquet  435751 B
     c9176329-eae5-4e14-a004-4ae66270f7bd-0_0-249-840_20260721050959743.parquet  435696 B
Hudi CoW: {'fb': 1, 'fa': 2, 'bb': 435751, 'ba': 871447, 'xtra': 0, 'kind': 'log'}


## Config 4 — Hudi merge-on-read

`type='mor'`: the `MERGE` appends a compact **log file** (`.log.`) next to the base file — no base rewrite.

In [6]:
spark.sql(f"""
CREATE TABLE {HUDI_MOR} (
    order_id BIGINT, customer_id BIGINT, amount DOUBLE, status STRING, updated_at TIMESTAMP
) USING hudi LOCATION '{HUDI_MOR_LOC}' TBLPROPERTIES (
    'primaryKey'='order_id', 'preCombineField'='updated_at', 'type'='mor'
)
""")
seed(HUDI_MOR)
h0 = hudi_health(HUDI_MOR_LOC)
do_merge(HUDI_MOR)
h1 = hudi_health(HUDI_MOR_LOC)
R["Hudi MoR"] = dict(fb=h0["parquet_files_on_disk"], fa=h1["parquet_files_on_disk"],
                     bb=h0["total_bytes"], ba=h1["total_bytes"], xtra=h1["log_files"], kind="log")
hudi_show_files(HUDI_MOR_LOC, "Hudi MoR after merge")
print("Hudi MoR:", R["Hudi MoR"])

  Hudi MoR after merge — data/log files on s3a://warehouse/hudi/lak12_mor:
     .1ea4089b-2dd6-49f3-89cd-1d20c383b00f-0_20260721051005095.log.1_0-298-1013     937 B
     1ea4089b-2dd6-49f3-89cd-1d20c383b00f-0_0-276-874_20260721051002256.parquet  435753 B
Hudi MoR: {'fb': 1, 'fa': 1, 'bb': 435753, 'ba': 436690, 'xtra': 1, 'kind': 'log'}


## Prove it — the cost of the same 1-row upsert, four ways

Read the **Δbytes** column: that's the write amplification. Copy-on-write pays a big byte delta (it rewrote a base file); merge-on-read pays a small one (it wrote a side file). File *counts* alone would hide this — the two Iceberg rows can show the same count while their byte deltas differ by an order of magnitude.

In [7]:
def kb(n): return f"{n/1024:.1f}KB"

print("=" * 92)
print("LAK-12  ·  one 1-row MERGE, four table configs  ·  Δbytes = write amplification")
print("=" * 92)
hdr = f"| {'Config':<11} | {'files b→a':<11} | {'bytes b→a':<17} | {'Δbytes':<9} | {'side files':<12} |"
print(hdr); print("|" + "-"*13 + "|" + "-"*13 + "|" + "-"*19 + "|" + "-"*11 + "|" + "-"*14 + "|")
for cfg in ("Iceberg CoW", "Iceberg MoR", "Hudi CoW", "Hudi MoR"):
    r = R[cfg]
    dbytes = r["ba"] - r["bb"]
    side = f"{r['xtra']} {r['kind']}"
    print(f"| {cfg:<11} | {str(r['fb'])+'→'+str(r['fa']):<11} | "
          f"{kb(r['bb'])+'→'+kb(r['ba']):<17} | {'+'+kb(dbytes):<9} | {side:<12} |")

print("\nIceberg snapshot detail (the last MERGE's own summary):")
for cfg in ("Iceberg CoW", "Iceberg MoR"):
    s = R[cfg]["snap"]
    print(f"  {cfg}: added-data={s['added']} deleted-data={s['deleted']} added-delete={s['added_deletes']}")

print("\nRow-count sanity (all four still 3 rows; order 1 → SHIPPED):")
for cfg, tbl in (("Iceberg CoW",ICE_COW),("Iceberg MoR",ICE_MOR),("Hudi CoW",HUDI_COW),("Hudi MoR",HUDI_MOR)):
    n = spark.sql(f"SELECT COUNT(*) n FROM {tbl}").first()["n"]
    s = spark.sql(f"SELECT status FROM {tbl} WHERE order_id=1").first()["status"]
    assert n == 3 and s == "SHIPPED", f"{cfg}: n={n} status={s}"
    print(f"  {cfg:<12} rows={n}  order 1 → {s}")
print("\nLAK-12 OK.")

LAK-12  ·  one 1-row MERGE, four table configs  ·  Δbytes = write amplification
| Config      | files b→a   | bytes b→a         | Δbytes    | side files   |
|-------------|-------------|-------------------|-----------|--------------|
| Iceberg CoW | 1→2         | 1.6KB→3.2KB       | +1.6KB    | 0 delete     |
| Iceberg MoR | 1→2         | 1.6KB→3.1KB       | +1.5KB    | 1 delete     |
| Hudi CoW    | 1→2         | 425.5KB→851.0KB   | +425.5KB  | 0 log        |
| Hudi MoR    | 1→1         | 425.5KB→426.5KB   | +0.9KB    | 1 log        |

Iceberg snapshot detail (the last MERGE's own summary):
  Iceberg CoW: added-data=1 deleted-data=1 added-delete=0
  Iceberg MoR: added-data=1 deleted-data=0 added-delete=1

Row-count sanity (all four still 3 rows; order 1 → SHIPPED):
  Iceberg CoW  rows=3  order 1 → SHIPPED
  Iceberg MoR  rows=3  order 1 → SHIPPED
  Hudi CoW     rows=3  order 1 → SHIPPED
  Hudi MoR     rows=3  order 1 → SHIPPED

LAK-12 OK.


## What you just saw

- **CoW and MoR are a write-vs-read trade, and it exists in *both* Iceberg and Hudi** — same idea, different plumbing (Iceberg delete files vs Hudi log files).
- **Measure amplification in bytes.** The `Δbytes` column separates CoW (big — rewrote a base file) from MoR (small — wrote a side file). File counts alone mislead.
- **MoR isn't free** — the cost moves to *read* time (reconciling deletes/logs) and to a later **compaction** job. CoW front-loads the cost so reads stay simple.

### Try it yourself
- Bump the seed to thousands of rows and re-run — watch the CoW Δbytes balloon while MoR stays flat.
- Add a second `MERGE` to the MoR tables and watch side files accumulate (that's the read tax growing until compaction).

**Related:** [LAK-8 (Iceberg MERGE)](../lak8_merge.ipynb) · [LAK-11 (Hudi intro)](./lak11_hudi_intro.ipynb).